In [11]:
import os
import re
import ast
import random
import time
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from collections import Counter

from nltk.translate.bleu_score import corpus_bleu

In [12]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")

print("Device:", device)

Device: cpu


In [13]:
TRAIN_PATH = "../en_fr_train.csv"
VAL_PATH   = "../en_fr_val.csv"
TEST_PATH  = "../en_fr_test.csv"

In [14]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (232825, 2)
Validation shape: (890, 2)
Test shape: (8597, 2)


In [15]:
print("Train columns:")
print(train_df.columns.tolist())

print("\nFirst 5 training rows:")
display(train_df.head())

Train columns:
['en', 'fr']

First 5 training rows:


,en,fr
0,"Thank you so much, Chris. And it's truly a gre...","Merci beaucoup, Chris. C'est vraiment un honne..."
1,"I have been blown away by this conference, and...",J'ai été très impressionné par cette conférenc...
2,"And I say that sincerely, partly because Put ...","Et je dis çà sincèrement, en autres parce que ..."
3,I flew on Air Force Two for eight years.,J'ai volé avec Air Force 2 pendant huit ans.
4,Now I have to take off my shoes or boots to ge...,"Et maintenant, je dois enlever mes chaussures ..."


In [16]:
print("Validation:")
display(val_df.head())

print("\nTest:")
display(test_df.head())

Validation:


,en,fr
0,Last year I showed these two slides so that d...,"L'année dernière, je vous ai présenté ces deux..."
1,But this understates the seriousness of this p...,Mais ceci tend à amoindrir le problème parce q...
2,"The arctic ice cap is, in a sense, the beatin...",On peut voir la calotte glacière arctique comm...
3,It expands in winter and contracts in summer.,Elle s'étend en hiver et se contracte en été.
4,The next slide I show you will be a rapid fas...,La prochaine diapositive que je vais vous mont...



Test:


,en,fr
0,"Several years ago here at TED, Peter Skillman ...","Il y a plusieurs années, ici à Ted, Peter Skil..."
1,And the idea's pretty simple: Teams of four h...,Et l'idée est plutôt simple. Des équipes de qu...
2,The marshmallow has to be on top.,Le marshmallow doit être placé au sommet.
3,"And, though it seems really simple, it's actua...","Bien que cela semble vraiment simple, c'est en..."
4,"And so, I thought this was an interesting idea...","J'ai trouvé que c'était une idée intéressante,..."


In [18]:
SRC_COL = "en"
TRG_COL = "fr"

In [19]:
print(train_df[SRC_COL].iloc[0])
print(type(train_df[SRC_COL].iloc[0]))

print(train_df[TRG_COL].iloc[0])
print(type(train_df[TRG_COL].iloc[0]))

Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.
<class 'str'>
Merci beaucoup, Chris. C'est vraiment un honneur de pouvoir venir sur cette scène une deuxième fois. Je suis très reconnaissant.
<class 'str'>


In [20]:
def parse_tokens(x):

    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    x = str(x).strip()

    # Try parsing Python-list representation
    if x.startswith("[") and x.endswith("]"):

        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return [
                    str(token)
                    for token in parsed
                ]

        except Exception:
            pass

    # Otherwise treat as normal text
    return x.split()

In [21]:
train_df["src_tokens"] = train_df[SRC_COL].apply(parse_tokens)
train_df["trg_tokens"] = train_df[TRG_COL].apply(parse_tokens)

val_df["src_tokens"] = val_df[SRC_COL].apply(parse_tokens)
val_df["trg_tokens"] = val_df[TRG_COL].apply(parse_tokens)

test_df["src_tokens"] = test_df[SRC_COL].apply(parse_tokens)
test_df["trg_tokens"] = test_df[TRG_COL].apply(parse_tokens)

In [22]:
print("SOURCE:")
print(train_df["src_tokens"].iloc[0])

print("\nTARGET:")
print(train_df["trg_tokens"].iloc[0])

SOURCE:
['Thank', 'you', 'so', 'much,', 'Chris.', 'And', "it's", 'truly', 'a', 'great', 'honor', 'to', 'have', 'the', 'opportunity', 'to', 'come', 'to', 'this', 'stage', 'twice;', "I'm", 'extremely', 'grateful.']

TARGET:
['Merci', 'beaucoup,', 'Chris.', "C'est", 'vraiment', 'un', 'honneur', 'de', 'pouvoir', 'venir', 'sur', 'cette', 'scène', 'une', 'deuxième', 'fois.', 'Je', 'suis', 'très', 'reconnaissant.']


In [23]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"


train_df["trg_tokens"] = train_df["trg_tokens"].apply(
    lambda x: [SOS_TOKEN] + x + [EOS_TOKEN]
)

val_df["trg_tokens"] = val_df["trg_tokens"].apply(
    lambda x: [SOS_TOKEN] + x + [EOS_TOKEN]
)

test_df["trg_tokens"] = test_df["trg_tokens"].apply(
    lambda x: [SOS_TOKEN] + x + [EOS_TOKEN]
)

In [24]:
def build_vocab(token_lists):

    counter = Counter()

    for tokens in token_lists:
        counter.update(tokens)

    vocab = {
        PAD_TOKEN: 0,
        UNK_TOKEN: 1
    }

    for token, count in counter.items():

        if token not in vocab:
            vocab[token] = len(vocab)

    return vocab

In [25]:
SRC_VOCAB = build_vocab(
    train_df["src_tokens"]
)

TRG_VOCAB = build_vocab(
    train_df["trg_tokens"]
)

print(
    "Source vocabulary size:",
    len(SRC_VOCAB)
)

print(
    "Target vocabulary size:",
    len(TRG_VOCAB)
)

Source vocabulary size: 136217
Target vocabulary size: 169938


In [26]:
## Reverse vocabulary

SRC_ID_TO_TOKEN = {
    idx: token
    for token, idx in SRC_VOCAB.items()
}

TRG_ID_TO_TOKEN = {
    idx: token
    for token, idx in TRG_VOCAB.items()
}

In [27]:
SRC_PAD_IDX = SRC_VOCAB[PAD_TOKEN]
SRC_UNK_IDX = SRC_VOCAB[UNK_TOKEN]

TRG_PAD_IDX = TRG_VOCAB[PAD_TOKEN]
TRG_UNK_IDX = TRG_VOCAB[UNK_TOKEN]

SOS_IDX = TRG_VOCAB[SOS_TOKEN]
EOS_IDX = TRG_VOCAB[EOS_TOKEN]

print("SRC PAD:", SRC_PAD_IDX)
print("TRG PAD:", TRG_PAD_IDX)
print("SOS:", SOS_IDX)
print("EOS:", EOS_IDX)

SRC PAD: 0
TRG PAD: 0
SOS: 2
EOS: 23


In [28]:
def encode_tokens(tokens, vocab):

    return [
        vocab.get(
            token,
            vocab[UNK_TOKEN]
        )
        for token in tokens
    ]

In [29]:
class TranslationDataset(Dataset):

    def __init__(self, dataframe):

        self.data = dataframe.reset_index(
            drop=True
        )

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        src_ids = encode_tokens(
            row["src_tokens"],
            SRC_VOCAB
        )

        trg_ids = encode_tokens(
            row["trg_tokens"],
            TRG_VOCAB
        )

        return (
            torch.tensor(
                src_ids,
                dtype=torch.long
            ),

            torch.tensor(
                trg_ids,
                dtype=torch.long
            )
        )

In [30]:
train_dataset = TranslationDataset(
    train_df
)

val_dataset = TranslationDataset(
    val_df
)

test_dataset = TranslationDataset(
    test_df
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 232825
Validation: 890
Test: 8597


In [31]:
def collate_fn(batch):

    src_batch, trg_batch = zip(*batch)

    max_src_len = max(
        len(x)
        for x in src_batch
    )

    max_trg_len = max(
        len(x)
        for x in trg_batch
    )

    padded_src = torch.full(
        (
            len(batch),
            max_src_len
        ),
        SRC_PAD_IDX,
        dtype=torch.long
    )

    padded_trg = torch.full(
        (
            len(batch),
            max_trg_len
        ),
        TRG_PAD_IDX,
        dtype=torch.long
    )

    for i, seq in enumerate(src_batch):

        padded_src[
            i,
            :len(seq)
        ] = seq

    for i, seq in enumerate(trg_batch):

        padded_trg[
            i,
            :len(seq)
        ] = seq

    # [batch, seq] -> [seq, batch]

    padded_src = padded_src.transpose(
        0,
        1
    )

    padded_trg = padded_trg.transpose(
        0,
        1
    )

    return padded_src, padded_trg

In [32]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [33]:
src, trg = next(
    iter(train_loader)
)

print("SRC shape:", src.shape)
print("TRG shape:", trg.shape)

SRC shape: torch.Size([82, 64])
TRG shape: torch.Size([76, 64])


In [34]:
INPUT_DIM = len(SRC_VOCAB)
OUTPUT_DIM = len(TRG_VOCAB)

ENC_EMB_DIM = 128
DEC_EMB_DIM = 128

HID_DIM = 128

DROPOUT = 0.1

LEARNING_RATE = 0.001

N_EPOCHS = 2

CLIP = 1.0

In [35]:
class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        emb_dim,
        hid_dim,
        dropout
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            emb_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.rnn = nn.GRU(
            emb_dim,
            hid_dim
        )

    def forward(self, src):

        embedded = self.dropout(
            self.embedding(src)
        )

        outputs, hidden = self.rnn(
            embedded
        )

        return outputs, hidden

In [37]:
class LuongAttention(nn.Module):

    def __init__(self, hid_dim):

        super().__init__()

        self.hid_dim = hid_dim

    def forward(
        self,
        decoder_hidden,
        encoder_outputs
    ):

        # encoder_outputs:
        # [src_len, batch, hid]

        # -> [batch, src_len, hid]

        encoder_outputs = encoder_outputs.permute(
            1,
            0,
            2
        )

        # decoder hidden:
        # [batch, hid]

        decoder_hidden = decoder_hidden.unsqueeze(
            2
        )

        # [batch, hid, 1]

        scores = torch.bmm(
            encoder_outputs,
            decoder_hidden
        ).squeeze(2)

        # [batch, src_len]

        attention_weights = torch.softmax(
            scores,
            dim=1
        )

        return attention_weights

In [38]:
class AttentionDecoder(nn.Module):

    def __init__(
        self,
        output_dim,
        emb_dim,
        hid_dim,
        attention,
        dropout
    ):

        super().__init__()

        self.output_dim = output_dim

        self.attention = attention

        self.embedding = nn.Embedding(
            output_dim,
            emb_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.rnn = nn.GRU(
            emb_dim + hid_dim,
            hid_dim
        )

        self.fc_out = nn.Linear(
            emb_dim + hid_dim + hid_dim,
            output_dim
        )

    def forward(
        self,
        input_token,
        hidden,
        encoder_outputs
    ):

        input_token = input_token.unsqueeze(
            0
        )

        embedded = self.dropout(
            self.embedding(input_token)
        )

        # Current decoder hidden state

        decoder_hidden = hidden[-1]

        # Attention

        attention_weights = self.attention(
            decoder_hidden,
            encoder_outputs
        )

        # [batch, src_len]
        # ->
        # [batch, 1, src_len]

        attention_weights = attention_weights.unsqueeze(
            1
        )

        encoder_outputs_batch = encoder_outputs.permute(
            1,
            0,
            2
        )

        # [batch, src_len, hid]

        context = torch.bmm(
            attention_weights,
            encoder_outputs_batch
        )

        # [batch, 1, hid]

        context = context.permute(
            1,
            0,
            2
        )

        # [1, batch, hid]

        rnn_input = torch.cat(
            (
                embedded,
                context
            ),
            dim=2
        )

        output, hidden = self.rnn(
            rnn_input,
            hidden
        )

        embedded = embedded.squeeze(0)
        output = output.squeeze(0)
        context = context.squeeze(0)

        prediction = self.fc_out(
            torch.cat(
                (
                    output,
                    context,
                    embedded
                ),
                dim=1
            )
        )

        return (
            prediction,
            hidden,
            attention_weights.squeeze(1)
        )

In [39]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder,
        device
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        batch_size = trg.shape[1]

        trg_len = trg.shape[0]

        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            trg_len,
            batch_size,
            trg_vocab_size,
            device=self.device
        )

        # Encoder

        encoder_outputs, hidden = self.encoder(
            src
        )

        # First decoder input = <sos>

        input_token = trg[0, :]

        for t in range(
            1,
            trg_len
        ):

            output, hidden, _ = self.decoder(
                input_token,
                hidden,
                encoder_outputs
            )

            outputs[t] = output

            teacher_force = (
                random.random()
                < teacher_forcing_ratio
            )

            top1 = output.argmax(
                dim=1
            )

            input_token = (
                trg[t]
                if teacher_force
                else top1
            )

        return outputs

In [40]:
attention = LuongAttention(
    HID_DIM
)

encoder = Encoder(
    INPUT_DIM,
    ENC_EMB_DIM,
    HID_DIM,
    DROPOUT
)

decoder = AttentionDecoder(
    OUTPUT_DIM,
    DEC_EMB_DIM,
    HID_DIM,
    attention,
    DROPOUT
)

model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)

print(
    f"Trainable parameters: "
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

Trainable parameters: 104,861,266


In [41]:
def initialize_weights(model):

    for name, param in model.named_parameters():

        if param.dim() > 1:

            nn.init.xavier_uniform_(
                param
            )

        else:

            nn.init.constant_(
                param,
                0
            )

initialize_weights(model)

In [42]:
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = nn.CrossEntropyLoss(
    ignore_index=TRG_PAD_IDX
)

In [43]:
def train_epoch(
    model,
    loader,
    optimizer,
    criterion,
    clip
):

    model.train()

    total_loss = 0

    for src, trg in loader:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(
            src,
            trg,
            teacher_forcing_ratio=0.5
        )

        output_dim = output.shape[-1]

        output = output[1:].reshape(
            -1,
            output_dim
        )

        trg = trg[1:].reshape(
            -1
        )

        loss = criterion(
            output,
            trg
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            clip
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [44]:
def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0

    with torch.no_grad():

        for src, trg in loader:

            src = src.to(device)
            trg = trg.to(device)

            output = model(
                src,
                trg,
                teacher_forcing_ratio=0
            )

            output_dim = output.shape[-1]

            output = output[1:].reshape(
                -1,
                output_dim
            )

            trg = trg[1:].reshape(
                -1
            )

            loss = criterion(
                output,
                trg
            )

            total_loss += loss.item()

    return total_loss / len(loader)

In [45]:
N_EPOCHS = 1

best_valid_loss = float("inf")

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        CLIP
    )

    valid_loss = evaluate(
        model,
        val_loader,
        criterion
    )

    elapsed = time.time() - start_time

    print(
        f"\nEpoch {epoch + 1}/{N_EPOCHS}"
    )

    print(
        f"Time: {elapsed / 60:.2f} minutes"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Validation Loss: {valid_loss:.4f}"
    )

    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss

        torch.save(
            model.state_dict(),
            "task2_attention_best.pt"
        )

        print(
            "✅ Best model saved"
        )

KeyboardInterrupt: 